In [1]:
from ssf.Constants import *
import pandas as pd

demo_dir = f"{DATA_DIR}/demographic_data"

files = ["rating-p-4.csv", "rating-p-5.csv", "hw-p-3.csv"]

for file in files:
  file_path = f"{demo_dir}/{file}"
  
  df = pd.read_csv(file_path)
  print(f"Loaded {file} with shape: {df.shape}")
  
  # Filter to keep only APPROVED rows
  df_approved = df[df['Status'] == 'APPROVED']
  print(f"After filtering for APPROVED status: {df_approved.shape}")
  
  # Display the first few rows of the filtered dataframe
  print(df_approved.head())

Loaded rating-p-4.csv with shape: (398, 21)
After filtering for APPROVED status: (319, 21)
              Submission id            Participant id    Status  \
0  682600904aee28c48c2ca020  675c5393b9b2d381baa880c2  APPROVED   
1  68260090f0e9d5d26bf7940c  60da68b6430d90da3b4e9bf6  APPROVED   
2  6826009399b46add8418dbc2  66aa8d04719b0f5ef3433e9c  APPROVED   
3  68260094b8d6e2b8f7f4214a  681bdc1e16b44acc6f14c3b6  APPROVED   
4  682600a93bc0dd65aed92efe  6100b94f1750b9db274684c7  APPROVED   

  Custom study tncs accepted at                   Started at  \
0                Not Applicable  2025-05-15T14:56:21.320000Z   
1                Not Applicable  2025-05-15T14:56:23.128000Z   
2                Not Applicable  2025-05-15T14:56:25.846000Z   
3                Not Applicable  2025-05-15T14:56:45.194000Z   
4                Not Applicable  2025-05-15T14:56:44.517000Z   

                  Completed at                  Reviewed at  \
0  2025-05-15T15:04:02.667000Z  2025-05-22T17:32:54.069000

In [2]:
# Inspect data structure
df_sample = pd.read_csv(f"{demo_dir}/hw-p-3.csv")
df_sample_approved = df_sample[df_sample['Status'] == 'APPROVED']

print("Column names:")
print(df_sample_approved.columns.tolist())
print("\nAge column unique values:")
print(sorted(df_sample_approved['Age'].dropna().unique()))
print("\nEthnicity simplified column unique values:")
print(df_sample_approved['Ethnicity simplified'].dropna().unique())
print("\nAge data type and sample values:")
print(df_sample_approved['Age'].dtype)
print(df_sample_approved['Age'].head(10))

Column names:
['Submission id', 'Participant id', 'Status', 'Custom study tncs accepted at', 'Started at', 'Completed at', 'Reviewed at', 'Archived at', 'Time taken', 'Completion code', 'Total approvals', 'Fluent languages', 'Social-media', 'Age', 'Sex', 'Ethnicity simplified', 'Country of birth', 'Country of residence', 'Nationality', 'Language', 'Student status', 'Employment status']

Age column unique values:
['22', '24', '25', '26', '27', '29', '31', '32', '33', '35', '36', '37', '38', '39', '40', '41', '45', '47', '48', '52', '57', '59', '61', '80']

Ethnicity simplified column unique values:
['White' 'Asian' 'Mixed' 'Black' 'DATA_EXPIRED']

Age data type and sample values:
object
0     35
1     24
3     47
6     31
7     41
8     61
9     32
13    35
16    33
18    48
Name: Age, dtype: object


In [3]:
def categorize_age(age):
    """Convert age to age range category"""
    if pd.isna(age) or age in ['CONSENT_REVOKED', 'DATA_EXPIRED']:
        return 'Unknown'
    
    try:
        age_num = int(age)
        if 18 <= age_num < 25:  # 18-24 (inclusive lower, exclusive upper)
            return '18-24'
        elif 25 <= age_num < 35:  # 25-34
            return '25-34'
        elif 35 <= age_num < 45:  # 35-44
            return '35-44'
        elif 45 <= age_num < 55:  # 45-54
            return '45-54'
        elif 55 <= age_num < 65:  # 55-64
            return '55-64'
        elif 65 <= age_num < 75:  # 65-74
            return '65-74'
        else:
            return '75+'
    except (ValueError, TypeError):
        return 'Unknown'

def create_latex_demographics_table(title, demographics_data, total_participants, sort_by_age=False):
    """Generate LaTeX formatted demographics table"""
    latex_output = f"\\textbf{{{title}}} \\\\\n"
    
    if sort_by_age:
        # Sort ages chronologically from youngest to oldest
        age_order = ['18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75+', 'Unknown']
        ordered_items = [(age, demographics_data[age]) for age in age_order if age in demographics_data]
    else:
        # Sort by count (highest percentage first)
        ordered_items = sorted(demographics_data.items(), key=lambda x: x[1], reverse=True)
    
    for category, count in ordered_items:
        percentage = (count / total_participants) * 100
        latex_output += f"{percentage:.1f}\\% {category} \\\\\n"
    
    return latex_output + "\n"

# Process each demographic file
demo_dir = f"{DATA_DIR}/demographic_data"
files = ["hw-p-3.csv", "rating-p-4.csv", "rating-p-5.csv"]
all_latex_output = []

for filename in files:
    # Load and filter data
    file_path = f"{demo_dir}/{filename}"
    df = pd.read_csv(file_path)
    approved_participants = df[df['Status'] == 'APPROVED']
    
    print(f"\n=== {filename} ===")
    print(f"Total approved participants: {len(approved_participants)}")
    
    # Calculate demographics
    age_ranges = [categorize_age(age) for age in approved_participants['Age']]
    age_distribution = pd.Series(age_ranges).value_counts().to_dict()
    gender_distribution = approved_participants['Sex'].value_counts().to_dict()
    ethnicity_distribution = approved_participants['Ethnicity simplified'].value_counts().to_dict()
    
    # Generate LaTeX tables
    study_name = filename.replace('.csv', '').replace('-', ' ').title()
    all_latex_output.append(f"% Demographics for {study_name}")
    
    gender_table = create_latex_demographics_table("Gender", gender_distribution, len(approved_participants))
    age_table = create_latex_demographics_table("Age", age_distribution, len(approved_participants), sort_by_age=True)
    ethnicity_table = create_latex_demographics_table("Race/Ethnicity", ethnicity_distribution, len(approved_participants))
    
    all_latex_output.extend([gender_table, age_table, ethnicity_table])
    
    # Display results
    print("\nGender distribution:")
    for gender, count in sorted(gender_distribution.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / len(approved_participants)) * 100
        print(f"  {percentage:.1f}% {gender}")
    
    print("\nAge distribution:")
    age_order = ['18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75+', 'Unknown']
    for age_range in age_order:
        if age_range in age_distribution:
            count = age_distribution[age_range]
            percentage = (count / len(approved_participants)) * 100
            print(f"  {percentage:.1f}% {age_range}")
    
    print("\nEthnicity distribution:")
    for ethnicity, count in sorted(ethnicity_distribution.items(), key=lambda x: x[1], reverse=True):
        percentage = (count / len(approved_participants)) * 100
        print(f"  {percentage:.1f}% {ethnicity}")

# Output all LaTeX tables
print("\n" + "="*50)
print("LATEX TABLES")
print("="*50)
for table in all_latex_output:
    print(table)


=== hw-p-3.csv ===
Total approved participants: 39

Gender distribution:
  51.3% Female
  48.7% Male

Age distribution:
  5.1% 18-24
  25.6% 25-34
  38.5% 35-44
  17.9% 45-54
  10.3% 55-64
  2.6% 75+

Ethnicity distribution:
  79.5% White
  10.3% Black
  5.1% Asian
  2.6% Mixed
  2.6% DATA_EXPIRED

=== rating-p-4.csv ===
Total approved participants: 319

Gender distribution:
  50.2% Female
  49.8% Male

Age distribution:
  11.9% 18-24
  17.9% 25-34
  16.6% 35-44
  15.7% 45-54
  27.9% 55-64
  9.1% 65-74
  0.9% 75+

Ethnicity distribution:
  63.3% White
  11.6% Black
  11.0% Mixed
  7.5% Other
  6.6% Asian

=== rating-p-5.csv ===
Total approved participants: 110

Gender distribution:
  50.0% Female
  50.0% Male

Age distribution:
  4.5% 18-24
  20.9% 25-34
  30.0% 35-44
  25.5% 45-54
  10.9% 55-64
  7.3% 65-74
  0.9% 75+

Ethnicity distribution:
  71.8% White
  14.5% Black
  8.2% Asian
  2.7% Mixed
  1.8% DATA_EXPIRED
  0.9% Other

LATEX TABLES
% Demographics for Hw P 3
\textbf{Gender} 

In [4]:
# Save LaTeX tables to disk
output_file = f"{DEMOGRAPHIC_RESULTS_DIR}/demographic_tables.tex"
with open(output_file, 'w') as f:
    for table in all_latex_output:
        f.write(table)

print(f"LaTeX tables saved to {output_file}")

# Also save individual files for each study
for i, filename in enumerate(files):
    study_name = filename.replace('.csv', '').replace('-', '_').lower()
    individual_file = f"{DEMOGRAPHIC_RESULTS_DIR}/demographics_{study_name}.tex"
    
    # Each study has 4 items in all_latex_output: comment + 3 tables
    start_idx = i * 4
    end_idx = start_idx + 4
    
    with open(individual_file, 'w') as f:
        for table in all_latex_output[start_idx:end_idx]:
            f.write(table)
    
    print(f"Individual tables for {filename} saved to {individual_file}")

LaTeX tables saved to ../results/demographics/demographic_tables.tex
Individual tables for hw-p-3.csv saved to ../results/demographics/demographics_hw_p_3.tex
Individual tables for rating-p-4.csv saved to ../results/demographics/demographics_rating_p_4.tex
Individual tables for rating-p-5.csv saved to ../results/demographics/demographics_rating_p_5.tex


In [5]:
def create_combined_demographics_table(files_data):
    """Create a single LaTeX table with all three studies side by side"""
    
    # Get study names and participant counts
    study_info = []
    for filename, data in files_data.items():
        study_name = filename.replace('.csv', '').replace('-', ' ').title()
        participant_count = data['total_participants']
        study_info.append((study_name, participant_count))
    
    # Start LaTeX table with vertical lines between columns
    latex = """\\begin{table}[h]
\\centering
\\caption{Demographic Characteristics Across Studies}
\\begin{tabular}{|l|c|c|c|}
\\hline
"""
    
    # Create header row with study names and N counts
    header = "Demographic & "
    for study_name, count in study_info:
        header += f"{study_name} (N={count}) & "
    header = header.rstrip(" & ") + " \\\\\n\\hline\n"
    latex += header
    
    # Get all unique categories for each demographic type
    all_genders = set()
    all_ages = set()
    all_ethnicities = set()
    
    for data in files_data.values():
        all_genders.update(data['gender_distribution'].keys())
        all_ages.update(data['age_distribution'].keys())
        all_ethnicities.update(data['ethnicity_distribution'].keys())
    
    # Gender section
    latex += "\\textbf{Gender} & & & \\\\\n"
    gender_order = sorted(all_genders, key=lambda x: sum(data['gender_distribution'].get(x, 0) for data in files_data.values()), reverse=True)
    
    for gender in gender_order:
        row = f"{gender} & "
        for filename, data in files_data.items():
            count = data['gender_distribution'].get(gender, 0)
            percentage = (count / data['total_participants']) * 100 if count > 0 else 0
            row += f"{percentage:.1f}\\% & "
        row = row.rstrip(" & ") + " \\\\\n"
        latex += row
    
    latex += "\\hline\n"
    
    # Age section
    latex += "\\textbf{Age} & & & \\\\\n"
    age_order = ['18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75+', 'Unknown']
    
    for age_range in age_order:
        if any(age_range in data['age_distribution'] for data in files_data.values()):
            row = f"{age_range} & "
            for filename, data in files_data.items():
                count = data['age_distribution'].get(age_range, 0)
                percentage = (count / data['total_participants']) * 100 if count > 0 else 0
                row += f"{percentage:.1f}\\% & "
            row = row.rstrip(" & ") + " \\\\\n"
            latex += row
    
    latex += "\\hline\n"
    
    # Ethnicity section
    latex += "\\textbf{Race/Ethnicity} & & & \\\\\n"
    ethnicity_order = sorted(all_ethnicities, key=lambda x: sum(data['ethnicity_distribution'].get(x, 0) for data in files_data.values()), reverse=True)
    
    for ethnicity in ethnicity_order:
        row = f"{ethnicity} & "
        for filename, data in files_data.items():
            count = data['ethnicity_distribution'].get(ethnicity, 0)
            percentage = (count / data['total_participants']) * 100 if count > 0 else 0
            row += f"{percentage:.1f}\\% & "
        row = row.rstrip(" & ") + " \\\\\n"
        latex += row
    
    latex += """\\hline
\\end{tabular}
\\end{table}
"""
    
    return latex

# Collect all data for the combined table
files_data = {}
for filename in files:
    file_path = f"{demo_dir}/{filename}"
    df = pd.read_csv(file_path)
    approved_participants = df[df['Status'] == 'APPROVED']
    
    # Calculate demographics
    age_ranges = [categorize_age(age) for age in approved_participants['Age']]
    
    files_data[filename] = {
        'total_participants': len(approved_participants),
        'age_distribution': pd.Series(age_ranges).value_counts().to_dict(),
        'gender_distribution': approved_participants['Sex'].value_counts().to_dict(),
        'ethnicity_distribution': approved_participants['Ethnicity simplified'].value_counts().to_dict()
    }

# Generate combined table
combined_table = create_combined_demographics_table(files_data)

# Save combined table
with open(f"{DEMOGRAPHIC_RESULTS_DIR}/demographics_combined_table.tex", 'w') as f:
    f.write(combined_table)

print("Combined demographics table saved to demographics_combined_table.tex")
print("\n" + "="*50)
print("COMBINED TABLE PREVIEW")
print("="*50)
print(combined_table)

Combined demographics table saved to demographics_combined_table.tex

COMBINED TABLE PREVIEW
\begin{table}[h]
\centering
\caption{Demographic Characteristics Across Studies}
\begin{tabular}{|l|c|c|c|}
\hline
Demographic & Hw P 3 (N=39) & Rating P 4 (N=319) & Rating P 5 (N=110) \\
\hline
\textbf{Gender} & & & \\
Female & 51.3\% & 50.2\% & 50.0\% \\
Male & 48.7\% & 49.8\% & 50.0\% \\
\hline
\textbf{Age} & & & \\
18-24 & 5.1\% & 11.9\% & 4.5\% \\
25-34 & 25.6\% & 17.9\% & 20.9\% \\
35-44 & 38.5\% & 16.6\% & 30.0\% \\
45-54 & 17.9\% & 15.7\% & 25.5\% \\
55-64 & 10.3\% & 27.9\% & 10.9\% \\
65-74 & 0.0\% & 9.1\% & 7.3\% \\
75+ & 2.6\% & 0.9\% & 0.9\% \\
\hline
\textbf{Race/Ethnicity} & & & \\
White & 79.5\% & 63.3\% & 71.8\% \\
Black & 10.3\% & 11.6\% & 14.5\% \\
Mixed & 2.6\% & 11.0\% & 2.7\% \\
Asian & 5.1\% & 6.6\% & 8.2\% \\
Other & 0.0\% & 7.5\% & 0.9\% \\
DATA_EXPIRED & 2.6\% & 0.0\% & 1.8\% \\
\hline
\end{tabular}
\end{table}

